# Original Idiap / USSD teachers on canonical DAIC-WOZ TRAIN/DEV (seed 103)

Run sections in order. Both teacher branches load their released pretrained checkpoint, fine-tune on the same official TRAIN-107 participants, and select checkpoints on the same canonical DEV-34 participants. Participant 440 is excluded because its source files are corrupt. The official TEST-47 roster is recorded in the manifest, but its media and predictions remain unopened.

**Validation limitation:** the released checkpoints have prior DAIC-WOZ exposure, and the USSD checkpoint has previously reported DEV results. Treat DEV metrics as exploratory for checkpoint selection. Keep TEST closed until the method is fixed.

Outputs go to `DAIC_WOZ/experiments/idiap_ussd_canonical188_seed103`; prior experiments remain untouched.

## 1. Mount Drive and load the pinned GitHub branch


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
if not (Path('/content/drive/MyDrive/DAIC_WOZ') / 'metadata').is_dir():
    drive.mount('/content/drive')
import base64, datetime, json, os, subprocess, sys
DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
assert (DAIC / 'metadata').is_dir(), f'Drive unavailable; reconnect Drive in Colab, then rerun: {DAIC}'
REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
BRANCH = 'feature/official141-original-teachers-80-10-10'
ROOT = Path('/content/reliability-aware-depression-kd')
CODE = Path('/content/reliability-aware-depression-kd-original188')
token = userdata.get('GITHUB_TOKEN'); assert token, 'Add GITHUB_TOKEN to Colab Secrets'
header = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy(); env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader', GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {header}')
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', f'https://github.com/{REPO}.git', str(ROOT)], check=True, env=env)
subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True, env=env)
revision = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
if not CODE.exists():
    subprocess.run(['git', '-C', str(ROOT), 'worktree', 'add', '--detach', str(CODE), revision], check=True)
else:
    assert (CODE / '.git').exists(), f'Existing folder is not a Git worktree: {CODE}'


def safe_switch(target):
    """Switch this worktree, preserving dirty work before retrying a refused switch."""
    command = ['git', '-C', str(CODE), 'switch', '--detach', target]
    first = subprocess.run(command, text=True, capture_output=True)
    if first.returncode == 0:
        return
    status = subprocess.run(['git', '-C', str(CODE), 'status', '--porcelain', '--untracked-files=all'],
                            text=True, capture_output=True)
    if status.returncode != 0:
        raise RuntimeError(f'git switch failed:\n{first.stderr}\ngit status failed:\n{status.stderr}')
    if not status.stdout.strip():
        raise RuntimeError(f'git switch failed with a clean worktree. Git says:\n{first.stderr}')

    before = subprocess.run(['git', '-C', str(CODE), 'rev-parse', '-q', '--verify', 'refs/stash'],
                            text=True, capture_output=True)
    old_stash = before.stdout.strip() if before.returncode == 0 else ''
    stash_name = 'colab-preserved-before-switch-' + target[:12]
    saved = subprocess.run(['git', '-C', str(CODE), 'stash', 'push', '--include-untracked', '-m', stash_name],
                           text=True, capture_output=True)
    if saved.returncode != 0:
        raise RuntimeError(f'Git refused the switch and could not preserve local files.\n'
                           f'Switch error: {first.stderr}\nStash error: {saved.stderr}')
    current_stash = subprocess.run(['git', '-C', str(CODE), 'rev-parse', '-q', '--verify', 'refs/stash'],
                                   text=True, capture_output=True)
    stash_sha = current_stash.stdout.strip() if current_stash.returncode == 0 else ''
    if not stash_sha or stash_sha == old_stash:
        raise RuntimeError(f'Git refused the switch, and no new stash was created.\n'
                           f'Switch error: {first.stderr}\nStash output: {saved.stdout}{saved.stderr}')

    backup_root = DAIC / 'experiments' / 'colab_worktree_backups'
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    backup = backup_root / f'{stamp}_{target[:12]}'
    backup.mkdir(parents=True, exist_ok=False)
    patch = subprocess.run(['git', '-C', str(CODE), 'stash', 'show', '--patch', '--binary',
                            '--include-untracked', stash_sha], text=True, capture_output=True)
    if patch.returncode != 0 or not patch.stdout:
        raise RuntimeError(f'Local files remain safely in Git stash {stash_sha}, but Drive backup export failed. '
                           f'No switch was attempted. Details: {patch.stderr}')
    (backup / 'local_changes.patch').write_text(patch.stdout)
    (backup / 'git_status_before_switch.txt').write_text(status.stdout)
    (backup / 'manifest.json').write_text(json.dumps({
        'target_revision': target,
        'stash_commit': stash_sha,
        'stash_name': stash_name,
        'worktree': str(CODE),
        'original_switch_error': first.stderr,
        'backup_utc': stamp,
        'restore_note': 'Changes are retained in the worktree Git stash. Inspect the patch before applying it.'
    }, indent=2) + '\n')
    clean = subprocess.check_output(['git', '-C', str(CODE), 'status', '--porcelain', '--untracked-files=all'], text=True)
    if clean.strip():
        raise RuntimeError(f'Files were stashed and backed up to {backup}, but the worktree is still dirty; '
                           f'no switch was attempted. Remaining status:\n{clean}')
    retry = subprocess.run(command, text=True, capture_output=True)
    if retry.returncode != 0:
        raise RuntimeError(f'Local work is preserved in stash {stash_sha} and backup {backup}. '
                           f'The switch still failed after cleanup:\n{retry.stderr}')
    print(f'Preserved local work in Git stash {stash_sha} and Drive backup {backup}')


if subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip() != revision:
    safe_switch(revision)
assert subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip() == revision
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-colab.txt')], check=True)
del token, header, env
# Colab retains already imported modules from earlier revisions.
sys.path = [entry for entry in sys.path if not entry.startswith('/content/reliability-aware-depression-kd-original188')]
sys.path.insert(0, str(CODE))
for name in list(sys.modules):
    if name == 'scripts' or name.startswith('scripts.'):
        del sys.modules[name]
assert (CODE / 'scripts/experiments/internal188/coverage.py').is_file(), 'Checkout is missing coverage.py'
print('Revision:', revision)


def refresh_code():
    """Update this existing worktree; no new /content code folder."""
    secret = userdata.get('GITHUB_TOKEN')
    assert secret, 'GITHUB_TOKEN missing from Colab Secrets'
    credentials = os.environ.copy()
    credentials.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader',
                       GIT_CONFIG_VALUE_0='AUTHORIZATION: basic ' +
                       base64.b64encode(f'x-access-token:{secret}'.encode()).decode())
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True, env=credentials)
    latest = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
    current = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip()
    if latest != current:
        safe_switch(latest)
    sys.path = [entry for entry in sys.path if not entry.startswith('/content/reliability-aware-depression-kd-original188')]
    sys.path.insert(0, str(CODE))
    for name in list(sys.modules):
        if name == 'scripts' or name.startswith('scripts.'):
            del sys.modules[name]
    print('Code revision:', latest)
    del secret, credentials


## 2. Freeze the canonical 107 / 34 / 47 split


In [ ]:
import pandas as pd
if str(CODE) not in sys.path: sys.path.insert(0, str(CODE))
from scripts.experiments.internal188.official_split import main as split_stage
EXP = DAIC / 'experiments' / 'idiap_ussd_canonical188_seed103'
split_stage(['--daic-root', str(DAIC), '--output', str(EXP / 'split')])
manifest = pd.read_csv(EXP / 'split' / 'manifest.csv')
assert len(manifest) == 188 and manifest.participant_id.nunique() == 188
assert 440 not in set(manifest.participant_id)
assert manifest.split.value_counts().to_dict() == {'train': 107, 'val': 34, 'student_test': 47}
print(pd.crosstab(manifest.split, manifest.label))
print('Canonical source audit:', manifest.groupby(['split', 'source_split']).size().to_dict())
print('TEST media opened: False; manifest:', EXP / 'split' / 'manifest.csv')


## 3. Evaluate the frozen Step-Audio2 LoRA adapter on official TRAIN/DEV

Use the frozen `maimai11/woz` adapter on one full original DAIC interview WAV per participant: TRAIN-107 and DEV-34. The earlier 20-second chunking is not used. The adapter card discloses `max_length=8000`, no forced response prefix, `temperature=0`, and `max_new_tokens=64`; its original preprocessing JSONL/code is not public, so this is the closest reproducible evaluation of the published setup, not a claim of exact preprocessing replication. TEST-47 stays unopened.


In [ ]:
refresh_code()
import json, os, shutil, subprocess, sys
from scripts.experiments.internal188.stepaudio2_adapter_audit import main as adapter_audit_stage
from scripts.experiments.internal188.prepare_stepaudio2_official_eval import main as prepare_stepaudio2_eval
from scripts.experiments.internal188.evaluate_stepaudio2_official import main as evaluate_stepaudio2_eval
from scripts.experiments.internal188.run_logged import run as run_logged
STEP_AUDIT = EXP / 'teachers' / 'stepaudio2_official_adapter_audit'
adapter_audit_stage(['--split-dir', str(EXP / 'split'), '--output', str(STEP_AUDIT)])
STEP_EVAL = EXP / 'teachers' / 'stepaudio2_frozen_authorstyle_official'
command = prepare_stepaudio2_eval(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
    '--adapter-audit', str(STEP_AUDIT / 'audit.json'), '--output', str(STEP_EVAL)])
gpu_name = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
if shutil.which('swift') is None: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ms-swift'])
if 'T4' in gpu_name:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes'])
    plugin = CODE / 'scripts/experiments/internal188/stepaudio2_t4_fp16_plugin.py'
    assert plugin.is_file(), f'Missing T4 plugin: {plugin}'
    command += ['--quant_method', 'bnb', '--quant_bits', '8', '--torch_dtype', 'float16', '--external_plugins', str(plugin)]
else: command += ['--torch_dtype', 'bfloat16']
env = os.environ.copy(); env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
run_logged(command, STEP_EVAL / 'swift_infer.log', cwd=CODE, env=env)
metrics = evaluate_stepaudio2_eval(['--split-dir', str(EXP / 'split'),
    '--index', str(STEP_EVAL / 'train_dev_index.csv'), '--generations', str(STEP_EVAL / 'train_dev_predictions.jsonl'),
    '--output', str(STEP_EVAL / 'evaluation')])
print('Frozen Step-Audio2 results:', STEP_EVAL / 'evaluation' / 'metrics.json')


## 4. Audit ComParE16 coverage for Idiap and USSD teachers

This separate audit checks ComParE16 features and transcripts for the 141 TRAIN/DEV participants. Step-Audio2 above uses the original full audio files and does not depend on this cache. The 47 TEST audio and transcript files are not opened.


In [ ]:
from scripts.experiments.internal188.coverage import main as coverage_stage
cache_candidates = [DAIC / 'experiments' / 'ussd_compare16',
                    DAIC / 'experiments' / 'full188_seed42' / 'features',
                    DAIC / 'experiments' / 'full188_80_10_10_seed103' / 'features']
available = [p for p in cache_candidates if (p / 'participant_manifest.csv').is_file()]
print('Available feature caches:', [str(p) for p in available])
assert available, 'No ComParE16 feature manifest found'
coverage_stage(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
                '--output', str(EXP / 'coverage'),
                *[arg for p in available for arg in ('--feature-cache', str(p))]])
covered = pd.read_csv(EXP / 'coverage' / 'participant_manifest.csv')
assert covered.split.value_counts().to_dict() == {'train': 107, 'val': 34}
print('Coverage:', pd.crosstab(covered.split, covered.label))


## 5. Fine-tune the original Idiap text teacher
This loads the published Idiap checkpoint and vectorizer, transfers compatible weights into the rebuilt TRAIN-107 graph, fine-tunes on TRAIN, and selects by DEV-34 macro F1. It writes epoch metrics, confusion matrices, logits, probabilities, and embeddings. TEST remains unopened.


In [ ]:
refresh_code()
assert (CODE / 'scripts/experiments/internal188/idiap_source.py').is_file(), 'Stale CODE checkout: rerun notebook section 1 before training'
from scripts.experiments.internal188.idiap_source import ensure_source, ensure_published_weights
from scripts.experiments.internal188.train_text_teacher import main as text_teacher_stage
idiap_source = ensure_source(DAIC.parent / 'tools' / 'solo_teacher_sources' / 'bias_in_daic-woz')
checkpoint, old_vectorizer = ensure_published_weights()
text_teacher_stage(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
                    '--coverage-dir', str(EXP / 'coverage'),
                    '--idiap-source', str(idiap_source),
                    '--published-checkpoint', str(checkpoint),
                    '--published-vectorizer', str(old_vectorizer),
                    '--output', str(EXP / 'teachers' / 'text_published_init'), '--seed', '103'])


In [ ]:
import json
pretrained_audit = json.loads((EXP / 'teachers' / 'text_published_init' / 'audit.json').read_text())
print('Transferred:', pretrained_audit['initialization'])
print('Prior exposure:', pretrained_audit['published_checkpoint_prior_exposure'])
print('Best epoch:', pretrained_audit['best_epoch'])
print('TRAIN:', {k: pretrained_audit['train'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})
print('VAL:', {k: pretrained_audit['val'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})


## 6. Fine-tune the original USSD audio teacher
After reviewing the frozen checkpoint baseline, this cell fine-tunes the same pinned USSD ComParE16+LSTM checkpoint on the same TRAIN-107 participants and selects on DEV-34. It saves epoch-level TRAIN/DEV F1 plus participant logits, probabilities, and embeddings. TEST remains unopened.


In [ ]:
refresh_code()
from scripts.teachers.ussd_audio.common import _resolve_pinned_author_artifact, AUTHOR_RUN4_CHECKPOINT_SHA256
from scripts.teachers.ussd_audio import CHECKPOINT_NAME
from scripts.experiments.internal188.train_audio_teacher import main as audio_teacher_stage
ussd_checkpoint = _resolve_pinned_author_artifact(
    DAIC.parent / 'tools' / 'solo_teacher_sources' / 'USSD-depression',
    CHECKPOINT_NAME, AUTHOR_RUN4_CHECKPOINT_SHA256)
audio_teacher_stage(['--split-dir', str(EXP / 'split'), '--coverage-dir', str(EXP / 'coverage'),
                     '--features', str(EXP / 'coverage'), '--published-checkpoint', str(ussd_checkpoint),
                     '--author-root', str(DAIC.parent / 'tools' / 'solo_teacher_sources' / 'USSD-depression'),
                     '--output', str(EXP / 'teachers' / 'audio_published_author_norm_v2'), '--seed', '103'])


In [ ]:
import json
audio_audit = json.loads((EXP / 'teachers' / 'audio_published_author_norm_v2' / 'audit.json').read_text())
print('Epoch 0 VAL:', {k: audio_audit['epoch_zero']['val'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})
print('Audio best epoch:', audio_audit['best_epoch'])
print('Prior exposure:', audio_audit['published_checkpoint_prior_exposure'])
print('TRAIN:', {k: audio_audit['train'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})
print('VAL:', {k: audio_audit['val'][k] for k in ('macro_f1', 'depressed_f1', 'auroc', 'confusion_matrix')})


## Stop before TEST

Review frozen Step-Audio2 metrics and both fine-tuned teacher results. Step-Audio2 is already DAIC-WOZ adapted; its author dataset preprocessing files and training roster are unavailable, so describe the evaluation as exploratory. Do not use TEST-47 until the method is fixed.
